# 우리가 지금까지 배운 분류 모델
- 로지스틱 회귀
- 의사결정트리 -> 트리기반 앙상블 모델 (실무에서 맣이 씀) -> 전처리 편함, 비선형 계산이 가능 (성능이 좋다)
- knn (사실상 train 하는데 걸리는 시간이 거의 없음 -> 데이터 세팅정도?)

-> 블랙홀이 하나 있음.... 실무에서 사용하기 꺼려지는....

TMI

XAI(eXplainable-AI) -> 설명가능 인공지능

xAI -> 일론머스크 기업 이름

실시간 학습? -> Online Train
모델 경량화

In [1]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

In [2]:
# 0) 데이터 준비

data = load_wine()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 로지스틱 회귀 (Logistic Regression)

## 주요 하이퍼파라미터

- C (규제 강도의 역수)

  - 규제(regularization) 강도

  - 수식적으로는 λ(람다)의 역수

  - 값이 작을수록 → 규제가 강함

  - 값이 클수록 → 규제가 약함


- penalty

  - l2 (가장 일반적)

  - l1 (불필요한 feature를 0으로 만들 수 있음 → feature selection 효과)

- solver
  - 최적화 알고리즘
  - lbfgs	다중분류에 안정적
  - liblinear	작은 데이터 + l1 가능

In [3]:
# 1) 로지스틱 회귀

logreg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000, random_state=42))
])

logreg_params = {
    "model__C": [0.1, 1, 10],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs"]
}

logreg_gs = GridSearchCV(
    logreg_pipe,
    param_grid=logreg_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

logreg_gs.fit(X_train, y_train)

print("Best Params:", logreg_gs.best_params_)
print("Best CV Score:", logreg_gs.best_score_)

y_pred_log = logreg_gs.predict(X_test)
acc_log = accuracy_score(y_test, y_pred_log)

print("Logistic Regression Accuracy:", acc_log)

Best Params: {'model__C': 0.1, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
Best CV Score: 0.993103448275862
Logistic Regression Accuracy: 1.0


In [4]:
# 로지스틱 회귀의 꽃은 확률도 볼 수 있는 다는 점 (test 1개만 예시를 들어보면)
logreg_gs.predict_proba(X_test)[0]

array([0.98747431, 0.01074511, 0.00178058])

In [5]:
logreg_gs.predict(X_test)[0]

np.int64(0)

# 의사결정트리 (Decision Tree)

## 주요 하이퍼파라미터

- max_depth

  - 트리의 최대 깊이

  - 가장 중요한 하이퍼파라미터

  - 3~5를 추천


- min_samples_split

  - 노드를 나누기 위한 최소 샘플 수

  - 값이 클 수록 더 분할, 값이 클 수록 최대한 분할

  - 과적합인 경우 값을 크게


- min_samples_leaf

  - 리프 노드 최소 샘플 수
  - 1~4를 추천
  - 과적합인 경우 값을 크게


In [6]:
# 2) 의사결정트리

tree = DecisionTreeClassifier(random_state=42)

tree_params = {
    "max_depth": [None, 3, 5, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

tree_gs = GridSearchCV(
    tree,
    param_grid=tree_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

tree_gs.fit(X_train, y_train)

print("Best Params:", tree_gs.best_params_)
print("Best CV Score:", tree_gs.best_score_)

y_pred_tree = tree_gs.predict(X_test)
acc_tree = accuracy_score(y_test, y_pred_tree)

print("Decision Tree Accuracy:", acc_tree)

Best Params: {'max_depth': 3, 'min_samples_leaf': 2, 'min_samples_split': 2}
Best CV Score: 0.930295566502463
Decision Tree Accuracy: 1.0


# K-최근접 이웃 (K-Nearest Neighbor ; KNN)

## 주요 하이퍼파라미터

- n_neighbors (k값)

  - 사실상 k가 전부, 적을수록 과적합, 많을 수록 과소적합

- weights

  - uniform	동일 가중치 (데이터 간단하면)

  - distance	가까운 점에 더 큰 가중치 (경계가 복잡할 것 같을 때)

- p (거리 계산 방식)

  - 1	Manhattan (이상치에 덜 민감하게 거리를 계산하고 싶을 때)
  - 2	Euclidean (대부분 이거 선택)

In [7]:
# 3) KNN
knn_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())
])

knn_params = {
    "model__n_neighbors": [3, 5, 7, 9, 11],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

knn_gs = GridSearchCV(
    knn_pipe,
    param_grid=knn_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

knn_gs.fit(X_train, y_train)

print("Best Params:", knn_gs.best_params_)
print("Best CV Score:", knn_gs.best_score_)

y_pred_knn = knn_gs.predict(X_test)
acc_knn = accuracy_score(y_test, y_pred_knn)

print("KNN Accuracy:", acc_knn)


Best Params: {'model__n_neighbors': 9, 'model__p': 1, 'model__weights': 'uniform'}
Best CV Score: 0.9862068965517242
KNN Accuracy: 1.0
